# build_dataset.py — Exploration Notebook

**Goal:** See what each function in the pipeline actually produces, so the code stops being abstract.

Run cells top to bottom the first time. After that, jump around freely.

Each section follows the same shape:
1. **Setup line(s)** — pre-written. Just run the cell.
2. **Explore prompts** — comments telling you what to investigate.
3. **TODO markers** — where you write the Polars expression yourself.

---

**Polars cheatsheet:**
```python
df.shape                              # (rows, cols)
df.columns                            # list of column names
df.schema                             # column types
df.head(10)                           # first 10 rows
df.glimpse()                          # one line per column — great for wide frames
df.describe()                         # summary stats
df.select(["a", "b"])                 # pick columns
df.filter(pl.col("x") > 5)            # filter rows
df.group_by("a").agg(pl.len())        # count per group
df.sort("x", descending=True)         # sort
df["col"].n_unique()                  # distinct count
df.select(pl.col("^prefix_.*$"))      # regex column select
df.transpose()                        # flip rows/columns (useful for wide 1-row frames)
```

## Setup

Load the pipeline functions and configure Polars display options.

In [70]:
import sys
from pathlib import Path

# scripts/ isn't a package — add it to sys.path so we can import from it
project_root = Path.cwd()
if project_root.name != "ff_ai_assistant":
    project_root = project_root.parent  # in case jupyter started from notebooks/
sys.path.insert(0, str(project_root / "scripts"))

import polars as pl

from build_dataset import (
    DEFAULT_SCORING,
    SCORING_KEYS,
    add_ranks,
    aggregate_to_season,
    build_combined,
    build_id_bridge,
    build_roster_meta,
    compute_value_scores,
    load_adp,
    load_rosters,
    load_weekly_stats,
    resolve_adp_player_ids,
)

pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(50)
pl.Config.set_fmt_str_lengths(60)
pl.Config.set_tbl_width_chars(120)

print("Setup OK.")
print("SCORING_KEYS:", SCORING_KEYS)
print("DEFAULT_SCORING:", DEFAULT_SCORING)

Setup OK.
SCORING_KEYS: ['espn_standard', 'espn_half_ppr', 'espn_ppr', 'yahoo_standard', 'yahoo_half_ppr', 'yahoo_ppr', 'sleeper_standard', 'sleeper_half_ppr', 'sleeper_ppr']
DEFAULT_SCORING: sleeper_half_ppr


## Step 1 — Raw loaders

Three thin CSV readers. Before any transformation, what do the raw inputs look like?

In [71]:
weekly = load_weekly_stats()
adp = load_adp()
rosters = load_rosters()

import polars.selectors as cs
# Explore — answer each with one Polars expression:
#   - What's the shape of each frame?
#   - How many `fantasy_points_*` columns does `weekly` have?
#     Hint: use a list comprehension on weekly.columns
#   - What columns does `adp` have? Notice `is_dst` and `match_key` —
#     these are enrichments added inside load_adp(), not in the raw CSV.
#   - Show the first 5 rows of `adp` where `is_dst == True`.
#     These are the DSTs that get treated separately in the pipeline.

# TODO:
print(f"weekly shape: {weekly.shape}")
print(f"adp shape: {adp.shape}")
print(f"rosters shape: {rosters.shape}")

# Use selectors and list comprehensions to answer the questions about columns:
weekly_filtered = weekly.select(cs.contains("fantasy_points_"))
print(weekly_filtered.columns)

adp.columns

# Show the first 5 rows of `adp` where `is_dst == True`.
adp.filter(pl.col("is_dst") == True).head(5)

weekly shape: (52207, 124)
adp shape: (3205, 7)
rosters shape: (24862, 36)
['fantasy_points_ppr', 'fantasy_points_espn_standard', 'fantasy_points_espn_half_ppr', 'fantasy_points_espn_ppr', 'fantasy_points_yahoo_standard', 'fantasy_points_yahoo_half_ppr', 'fantasy_points_yahoo_ppr', 'fantasy_points_sleeper_standard', 'fantasy_points_sleeper_half_ppr', 'fantasy_points_sleeper_ppr']


season,player_name,adp,pos,team,is_dst,match_key
i64,str,f64,str,str,bool,str
2018,"""Jacksonville Jaguars""",95.0,"""DST""","""DST""",true,"""jacksonville jaguars"""
2018,"""Los Angeles Rams""",106.0,"""DST""","""DST""",true,"""los angeles rams"""
2018,"""Minnesota Vikings""",111.0,"""DST""","""DST""",true,"""minnesota vikings"""
2018,"""Philadelphia Eagles""",123.0,"""DST""","""DST""",true,"""philadelphia eagles"""
2018,"""Los Angeles Chargers""",127.0,"""DST""","""DST""",true,"""los angeles chargers"""


## Step 2 — Aggregate weekly to season

`aggregate_to_season` groups weekly rows into one row per `(player_id, season)`.
The loop over `SCORING_KEYS` generates one column per scoring preset automatically —
this is the "9-variant generalization."

In [72]:
season_stats = aggregate_to_season(weekly)

# Explore:
#   - Verify the grain. Compare:
#       season_stats.select(["player_id", "season"]).n_unique()
#     against:
#       season_stats.height
#     If they match, the grain invariant holds here.
#   - Pick one player (e.g. "Patrick Mahomes") and show all their rows
#     across seasons:
#       season_stats.filter(pl.col("player_display_name") == "...")
#   - For one player-season row, compare
#       seasonal_fantasy_points_sleeper_half_ppr
#     against
#       seasonal_fantasy_points_espn_standard
#     Why are they different? Check config.py if you need to.

# TODO:
print(f"season_stats height: {season_stats.height}")
print(f"Unique player-season combinations: {season_stats.select(['player_id', 'season']).n_unique()}")

cmc_stats = season_stats.filter(pl.col("player_display_name") == "Christian McCaffrey")
print(cmc_stats)

cmc_stats.select(["season","seasonal_fantasy_points_sleeper_half_ppr", "seasonal_fantasy_points_espn_standard"]).sort("season")


season_stats height: 5056
Unique player-season combinations: 5056
shape: (8, 25)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pla ┆ pla ┆ pos ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ gam ┆ tea ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ mat │
│ yer ┆ yer ┆ iti ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ es_ ┆ m   ┆ _es ┆ _es ┆ _es ┆ _ya ┆ _ya ┆ _ya ┆ _sl ┆ _sl ┆ _sl ┆ ch_ │
│ _id ┆ _di ┆ on  ┆ --- ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ pla ┆ --- ┆ pn_ ┆ pn_ ┆ pn_ ┆ hoo ┆ hoo ┆ hoo ┆ eep ┆ eep ┆ eep ┆ key │
│ --- ┆ spl ┆ --- ┆ i64 ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ yed ┆ str ┆ sta ┆ hal ┆ ppr ┆ _st ┆ _ha ┆ _pp ┆ er_ ┆ er_ ┆ er_ ┆ --- │
│ str ┆ ay_ ┆ str ┆     ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ --- ┆     ┆ nda ┆ f_p ┆ --- ┆ and ┆ lf_ ┆ r   ┆ sta ┆ hal ┆ ppr ┆ str │
│     ┆

season,seasonal_fantasy_points_sleeper_half_ppr,seasonal_fantasy_points_espn_standard
i64,f64,f64
2018,332.0,278.5
2019,413.2,355.2
2020,81.9,73.4
2021,109.0,90.5
2022,298.46,257.46
2023,357.8,324.3
2024,40.3,32.8
2025,356.9,308.9


## Step 3 — Ranks

`add_ranks` uses the `DEFAULT_SCORING` column (`sleeper_half_ppr`) to compute
overall and positional finish ranks. `method="min"` means ties share the lower rank.

In [73]:
season_stats = add_ranks(season_stats)

# Explore:
#   - Show the top 5 overall scorers in 2024
#     (season == 2024 and overall_points_rank <= 5).
#   - Show the #1 player at each fantasy position in 2024.
#   - method="min" was used. What happens to ties?
#     Find a season-position group where two players share a rank.

print(f"Top 5 overall scorers in 2024:")
print(season_stats.filter((pl.col("season") == 2024) & (pl.col("overall_points_rank") <= 5)).sort("overall_points_rank"))

print(f"#1 player at each fantasy position in 2024:")
print(season_stats.filter((pl.col("season") == 2024) & (pl.col("position_points_rank") == 1)).select(["player_display_name", "position", "position_points_rank", "season"]))


Top 5 overall scorers in 2024:
shape: (5, 27)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ pla ┆ pla ┆ pos ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ sea ┆ gam ┆ tea ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ ppg ┆ mat ┆ ove ┆ pos │
│ yer ┆ yer ┆ iti ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ son ┆ es_ ┆ m   ┆ _es ┆ _es ┆ _es ┆ _ya ┆ _ya ┆ _ya ┆ _sl ┆ _sl ┆ _sl ┆ ch_ ┆ ral ┆ iti │
│ _id ┆ _di ┆ on  ┆ --- ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ al_ ┆ pla ┆ --- ┆ pn_ ┆ pn_ ┆ pn_ ┆ hoo ┆ hoo ┆ hoo ┆ eep ┆ eep ┆ eep ┆ key ┆ l_p ┆ on_ │
│ --- ┆ spl ┆ --- ┆ i64 ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ fan ┆ yed ┆ str ┆ sta ┆ hal ┆ ppr ┆ _st ┆ _ha ┆ _pp ┆ er_ ┆ er_ ┆ er_ ┆ --- ┆ oin ┆ poi │
│ str ┆ ay_ ┆ str ┆     ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ tas ┆ --- ┆     ┆ nda ┆ f_p ┆ --- ┆ and ┆ lf_ ┆ r   ┆ sta ┆ 

## Step 4 — The ID bridge

`build_id_bridge` pulls nflreadpy's `ff_playerids` table — a master list that maps
player names (via `match_key`) to `gsis_id`. It deliberately preserves duplicate
`match_key` rows; disambiguation happens in Step 5.

> ⚠️ First run hits nflreadpy's cache — may take a few seconds.

In [74]:
bridge = build_id_bridge()

# Explore:
#   - How many total rows? How many unique match_keys?
#     The difference tells you how many names have multiple candidate players.
#   - Find ALL match_keys that appear more than once:
#       bridge.group_by("match_key").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
#   - Pick one duplicate match_key and look at the full bridge rows for it.
#     Are these actually different people, or data artifacts?

print(f"Total rows in bridge: {bridge.height}")
print(f"Unique match_keys in bridge: {bridge.select('match_key').n_unique()}")
duplicate_match_keys = bridge.group_by("match_key").agg(pl.len().alias("n")).filter(pl.col("n") > 1)
print(duplicate_match_keys)
print(bridge.filter(pl.col("match_key") == duplicate_match_keys[0, "match_key"]))
print(bridge.filter(pl.col("match_key") == duplicate_match_keys[1, "match_key"]))
print(bridge.filter(pl.col("match_key") == duplicate_match_keys[2, "match_key"]))
print(bridge.filter(pl.col("match_key") == duplicate_match_keys[3, "match_key"]))
print(bridge.filter(pl.col("match_key") == duplicate_match_keys[4, "match_key"]))

# Look at example for Lamar Jackson
print(bridge.filter(pl.col("match_key") == "lamar jackson"))

Total rows in bridge: 7715
Unique match_keys in bridge: 7584
shape: (112, 2)
┌────────────────┬─────┐
│ match_key      ┆ n   │
│ ---            ┆ --- │
│ str            ┆ u32 │
╞════════════════╪═════╡
│ dj moore       ┆ 2   │
│ ray agnew      ┆ 2   │
│ taiwan jones   ┆ 2   │
│ josh thomas    ┆ 2   │
│ tony brown     ┆ 3   │
│ michael thomas ┆ 2   │
│ kyle williams  ┆ 3   │
│ duke williams  ┆ 2   │
│ chris clemons  ┆ 2   │
│ josh harris    ┆ 2   │
│ …              ┆ …   │
│ tyler davis    ┆ 2   │
│ chris smith    ┆ 4   │
│ roy williams   ┆ 2   │
│ reggie walker  ┆ 2   │
│ ryan griffin   ┆ 2   │
│ justin jones   ┆ 2   │
│ mike brown     ┆ 2   │
│ marcus smith   ┆ 2   │
│ andre carter   ┆ 2   │
│ chris davis    ┆ 2   │
└────────────────┴─────┘
shape: (2, 5)
┌────────────┬────────────┬──────────┬────────────┬───────────┐
│ gsis_id    ┆ merge_name ┆ position ┆ name       ┆ match_key │
│ ---        ┆ ---        ┆ ---      ┆ ---        ┆ ---       │
│ str        ┆ str        ┆ str      ┆ str

## Step 5 — The 3-tier resolver

This is the smart part. `resolve_adp_player_ids` labels every ADP row with a `match_tier`:

| Tier | Meaning |
|---|---|
| `exact` | One bridge candidate — picked directly |
| `disambiguated` | Multiple bridge candidates — picked by "has stats that season" |
| `fuzzy` | No exact match — rapidfuzz found one above threshold |
| `unresolved` | Nothing worked |
| `dst` | Defense/ST — intentionally not resolved |

In [75]:
adp_resolved = resolve_adp_player_ids(adp, bridge, season_stats)
print(f"ADP resolved columns: {adp_resolved.columns}")
# Explore:
#   - Show the count per match_tier:
#       adp_resolved.group_by("match_tier").agg(pl.len())
#   - Pull a few rows from each of exact, disambiguated, and fuzzy.
#     Do the player_name values look right?
#   - Show ALL the fuzzy matches. Are any clearly wrong?
#   - Show ALL the unresolved rows. Why do you think they didn't match?

print(adp_resolved.group_by("match_tier").agg(pl.len()))
print(adp_resolved.filter(pl.col("match_tier") == "exact").select(["player_name", "match_key", "match_tier", "season"]).head(5))
print(adp_resolved.filter(pl.col("match_tier") == "disambiguated").select(["player_name", "match_key", "match_tier", "season"]).head(5))
print(adp_resolved.filter(pl.col("match_tier") == "fuzzy").select(["player_name", "match_key", "match_tier", "season"]).head(5))
print(adp_resolved.filter(pl.col("match_tier") == "unresolved").select(["player_name", "match_key", "match_tier", "season"]).head(5))

# Show all fuzzy matches
print("Fuzzy matches:\n")
print(adp_resolved.filter(pl.col("match_tier") == "fuzzy").select(["player_name", "match_key", "match_tier", "season"]))

# Show all unresolved matches
print("Unresolved matches:\n")
print(adp_resolved.filter(pl.col("match_tier") == "unresolved").select(["player_name", "match_key", "match_tier", "season"]))

ADP resolved columns: ['season', 'player_name', 'adp', 'pos', 'team', 'is_dst', 'match_key', 'gsis_id', 'bridge_position', 'match_tier']
shape: (5, 2)
┌───────────────┬──────┐
│ match_tier    ┆ len  │
│ ---           ┆ ---  │
│ str           ┆ u32  │
╞═══════════════╪══════╡
│ disambiguated ┆ 65   │
│ exact         ┆ 2803 │
│ fuzzy         ┆ 11   │
│ unresolved    ┆ 105  │
│ dst           ┆ 221  │
└───────────────┴──────┘
shape: (5, 4)
┌─────────────────┬─────────────────┬────────────┬────────┐
│ player_name     ┆ match_key       ┆ match_tier ┆ season │
│ ---             ┆ ---             ┆ ---        ┆ ---    │
│ str             ┆ str             ┆ str        ┆ i64    │
╞═════════════════╪═════════════════╪════════════╪════════╡
│ Skyy Moore      ┆ skyy moore      ┆ exact      ┆ 2023   │
│ Ricky Pearsall  ┆ ricky pearsall  ┆ exact      ┆ 2025   │
│ Knowshon Moreno ┆ knowshon moreno ┆ exact      ┆ 2018   │
│ DeAndre Hopkins ┆ deandre hopkins ┆ exact      ┆ 2025   │
│ Deuce Vaughn    ┆ 

## Step 6 — Combined table (the big join)

`build_combined` joins season stats + ADP + roster metadata, then appends
"holdouts" — ADP rows with no matching stats — as zero-stat rows.

Key design decision: **stats-first left join**. Stats are the left frame;
ADP is joined onto them. This guarantees the `(player_id, season)` grain.

In [83]:
roster_meta = build_roster_meta(rosters)
combined = build_combined(season_stats, adp_resolved, roster_meta)

# Explore:
#   - Verify the grain for rows with a non-null player_id:
#       with_id = combined.filter(pl.col("player_id").is_not_null())
#       with_id.select(["player_id", "season"]).n_unique() == with_id.height
#   - Count rows by category:
#       * has stats AND has adp  (drafted and played)
#       * has stats but null adp (played but undrafted / waiver pickups)
#       * games_played == 0      (holdouts — drafted but no stats)
#   - Look at 10 holdout rows for 2024. Are they players who retired,
#     got cut, or names the resolver missed?
#   - Pick a star player and show their full combined row for 2024.
#     Use .transpose() or .glimpse() to see all columns.

# Print the left and right frames that go into the final join, so we can understand the shape before the combine:
print(f"season_stats shape: {season_stats.columns}")
print(f"adp_resolved shape: {adp_resolved.columns}")
print(f"roster_meta shape: {roster_meta.columns}")

print(f"combined shape: {combined.shape}")

# 1. Verify grain on rows with a player_id
with_id = combined.filter(pl.col("player_id").is_not_null())
n_unique = with_id.select(["player_id", "season"]).n_unique()
print(f"\nGrain check (rows with player_id):")
print(f"  rows:            {with_id.height}")
print(f"  unique (id, season): {n_unique}")
print(f"  grain holds:     {n_unique == with_id.height}")

# 2. Row counts by category
drafted_and_played = combined.filter(
    (pl.col("games_played") > 0) & pl.col("adp").is_not_null()
).height
played_undrafted = combined.filter(
    (pl.col("games_played") > 0) & pl.col("adp").is_null()
).height
holdouts = combined.filter(pl.col("games_played") == 0).height
print(f"\nRow categories:")
print(f"  drafted and played:           {drafted_and_played}")
print(f"  played but undrafted:         {played_undrafted}")
print(f"  holdouts (drafted, no stats): {holdouts}")

# 3. Holdout rows for 2024
print("\n10 holdout rows for 2024 (sorted by ADP):")
print(
    combined.filter(
        (pl.col("games_played") == 0) & (pl.col("season") == 2024)
    )
    .select(["player_display_name", "position", "adp", "player_id"])
    .sort("adp")
    .head(10)
)

# 4. Full row for one star — Ja'Marr Chase 2024
print("\nJa'Marr Chase 2024 — full row (transposed):")
print(
    combined.filter(
        (pl.col("player_display_name") == "Ja'Marr Chase")
        & (pl.col("season") == 2024)
    ).transpose(include_header=True, header_name="column", column_names=["value"])
)

season_stats shape: ['player_id', 'player_display_name', 'position', 'season', 'seasonal_fantasy_points_espn_standard', 'seasonal_fantasy_points_espn_half_ppr', 'seasonal_fantasy_points_espn_ppr', 'seasonal_fantasy_points_yahoo_standard', 'seasonal_fantasy_points_yahoo_half_ppr', 'seasonal_fantasy_points_yahoo_ppr', 'seasonal_fantasy_points_sleeper_standard', 'seasonal_fantasy_points_sleeper_half_ppr', 'seasonal_fantasy_points_sleeper_ppr', 'games_played', 'team', 'ppg_espn_standard', 'ppg_espn_half_ppr', 'ppg_espn_ppr', 'ppg_yahoo_standard', 'ppg_yahoo_half_ppr', 'ppg_yahoo_ppr', 'ppg_sleeper_standard', 'ppg_sleeper_half_ppr', 'ppg_sleeper_ppr', 'match_key', 'overall_points_rank', 'position_points_rank']
adp_resolved shape: ['season', 'player_name', 'adp', 'pos', 'team', 'is_dst', 'match_key', 'gsis_id', 'bridge_position', 'match_tier']
roster_meta shape: ['gsis_id', 'season', 'roster_name', 'roster_team', 'roster_position', 'years_exp', 'birth_date', 'height', 'weight', 'college', 'd

## Step 7 — Value scores

`compute_value_scores` filters to matched QB/RB/WR/TE with ADP and computes:

- `pos_finish_rank` — where they actually finished (by fantasy points, within position-season)
- `adp_pos_rank` — where they were drafted (within position-season)
- `value_over_adp = adp_pos_rank - pos_finish_rank`
  - **Positive** = outperformed draft position (good value)
  - **Negative** = underperformed (bust)

In [77]:
analysis = compute_value_scores(combined)

# Explore:
#   - Top 10 value picks in 2024 (sort value_over_adp descending, filter season == 2024).
#   - Biggest busts in 2024 (sort ascending).
#   - Average value_over_adp by position in 2024. Which position is
#     hardest to predict at draft time?
#   - Find all player-seasons where value_over_adp > 50. Career-level outperformances.

VALUE_COLS = [
    "player_display_name", "position", "season",
    "adp", "adp_pos_rank", "pos_finish_rank", "value_over_adp",
]

# 1. Top 10 value picks in 2024
print("Top 10 value picks in 2024:")
print(
    analysis.filter(pl.col("season") == 2024)
    .sort("value_over_adp", descending=True)
    .select(VALUE_COLS)
    .head(10)
)

# 2. Biggest busts in 2024
print("\nBiggest busts in 2024:")
print(
    analysis.filter(pl.col("season") == 2024)
    .sort("value_over_adp")
    .select(VALUE_COLS)
    .head(10)
)

# 3. Average value_over_adp by position in 2024
# Note: position-by-position averages over a finite set of ranks tend toward 0;
# what matters is variance/spread (which positions are most volatile).
print("\nValue-over-ADP by position in 2024 (mean + spread):")
print(
    analysis.filter(pl.col("season") == 2024)
    .group_by("position")
    .agg(
        pl.col("value_over_adp").mean().round(2).alias("mean"),
        pl.col("value_over_adp").std().round(2).alias("std"),
        pl.col("value_over_adp").abs().mean().round(2).alias("avg_abs_miss"),
        pl.len().alias("n"),
    )
    .sort("avg_abs_miss", descending=True)
)

# 4. All player-seasons with value_over_adp > 50
print("\nCareer-level outperformances (value_over_adp > 50):")
print(
    analysis.filter(pl.col("value_over_adp") > 50)
    .select(VALUE_COLS)
    .sort("value_over_adp", descending=True)
)

Top 10 value picks in 2024:
shape: (10, 7)
┌─────────────────────┬──────────┬────────┬───────┬──────────────┬─────────────────┬────────────────┐
│ player_display_name ┆ position ┆ season ┆ adp   ┆ adp_pos_rank ┆ pos_finish_rank ┆ value_over_adp │
│ ---                 ┆ ---      ┆ ---    ┆ ---   ┆ ---          ┆ ---             ┆ ---            │
│ str                 ┆ str      ┆ i64    ┆ f64   ┆ i32          ┆ i32             ┆ i32            │
╞═════════════════════╪══════════╪════════╪═══════╪══════════════╪═════════════════╪════════════════╡
│ Alec Pierce         ┆ WR       ┆ 2024   ┆ 301.0 ┆ 104          ┆ 39              ┆ 65             │
│ Tre Tucker          ┆ WR       ┆ 2024   ┆ 290.5 ┆ 98           ┆ 53              ┆ 45             │
│ Jerry Jeudy         ┆ WR       ┆ 2024   ┆ 155.7 ┆ 56           ┆ 12              ┆ 44             │
│ Brian Thomas Jr.    ┆ WR       ┆ 2024   ┆ 111.3 ┆ 46           ┆ 4               ┆ 42             │
│ Rashod Bateman      ┆ WR       ┆ 2024

## Step 8 — Scoring generalization sandbox

`SCORING_KEYS = list(ALL_SCORING_SETTINGS.keys())` drives the entire multi-scoring
design. Every preset in `config.py` automatically produces its own seasonal column
via the loop in `aggregate_to_season`. Adding a 10th preset to `config.py` requires
zero changes in `build_dataset.py`.

In [78]:
# Explore:
#   - Print SCORING_KEYS. How many variants are there?
#   - Pull all seasonal_fantasy_points_* columns for one player in 2024:
#       combined.filter(
#           (pl.col("player_display_name") == "Justin Jefferson")
#           & (pl.col("season") == 2024)
#       ).select(pl.col("^seasonal_fantasy_points_.*$"))
#   - Which preset scores the top QB highest? Which scores them lowest?
#     Think: Yahoo uses 4-pt passing TDs; some presets use 6-pt.
#   - Trace where SCORING_KEYS is used in build_dataset.py. If you added
#     a 10th preset to config.py, what in this file would need to change?

print(f"Number of scoring variants: {len(SCORING_KEYS)}")
print(f"SCORING_KEYS: {SCORING_KEYS}")

# 1. Justin Jefferson 2024 across all scoring variants
print("\nJustin Jefferson 2024 — all seasonal_fantasy_points_* columns:")
print(
    combined.filter(
        (pl.col("player_display_name") == "Justin Jefferson")
        & (pl.col("season") == 2024)
    )
    .select(pl.col("^seasonal_fantasy_points_.*$"))
    .transpose(include_header=True, header_name="scoring", column_names=["points"])
    .sort("points", descending=True)
)

# 2. Top QB 2024 (Lamar) across all scoring variants — passing TD value matters here
print("\nLamar Jackson 2024 — all seasonal_fantasy_points_* columns:")
print(
    combined.filter(
        (pl.col("player_display_name") == "Lamar Jackson")
        & (pl.col("season") == 2024)
    )
    .select(pl.col("^seasonal_fantasy_points_.*$"))
    .transpose(include_header=True, header_name="scoring", column_names=["points"])
    .sort("points", descending=True)
)

# 3. Sanity check the design claim: PPR should always >= half_ppr >= standard
#    within a single platform family. Pick a high-volume WR.
print("\nJa'Marr Chase 2024 — within-family ordering check:")
chase = (
    combined.filter(
        (pl.col("player_display_name") == "Ja'Marr Chase")
        & (pl.col("season") == 2024)
    )
    .select(pl.col("^seasonal_fantasy_points_.*$"))
    .transpose(include_header=True, header_name="scoring", column_names=["points"])
    .sort("scoring")
)
print(chase)

Number of scoring variants: 9
SCORING_KEYS: ['espn_standard', 'espn_half_ppr', 'espn_ppr', 'yahoo_standard', 'yahoo_half_ppr', 'yahoo_ppr', 'sleeper_standard', 'sleeper_half_ppr', 'sleeper_ppr']

Justin Jefferson 2024 — all seasonal_fantasy_points_* columns:
shape: (9, 2)
┌──────────────────────────────────────────┬────────┐
│ scoring                                  ┆ points │
│ ---                                      ┆ ---    │
│ str                                      ┆ f64    │
╞══════════════════════════════════════════╪════════╡
│ seasonal_fantasy_points_espn_ppr         ┆ 309.08 │
│ seasonal_fantasy_points_yahoo_ppr        ┆ 309.08 │
│ seasonal_fantasy_points_sleeper_ppr      ┆ 309.08 │
│ seasonal_fantasy_points_espn_half_ppr    ┆ 259.08 │
│ seasonal_fantasy_points_yahoo_half_ppr   ┆ 259.08 │
│ seasonal_fantasy_points_sleeper_half_ppr ┆ 259.08 │
│ seasonal_fantasy_points_espn_standard    ┆ 209.08 │
│ seasonal_fantasy_points_yahoo_standard   ┆ 209.08 │
│ seasonal_fantasy_points

## Wrap-up — quiz yourself

When you can answer all 6 questions below **without looking at the code or walkthrough**,
you're ready for Session 1 (CI setup).

1. Why is it safe to assume `combined_stats_adp` has one row per `(player_id, season)`?
2. What goes wrong if the starting frame of `build_combined` is ADP instead of stats?
3. What's the difference between tier 2 (disambiguated) and tier 3 (fuzzy), and why is tier 2 run first?
4. Why does `seasonal_fantasy_points_sleeper_half_ppr` have slightly different values than what nflreadpy would give you directly?
5. Why do holdout rows get **zero-valued** seasonal stats instead of null?
6. What fails silently if you delete the `validate()` call?

If you're unsure on any of them, go back to the relevant step above and look harder.